# 01-9_remove_wcs_in_HDUL_ys

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [2]:
#import sys
%pip install photutils==1.6 astropy==5.2.1

Note: you may need to restart the kernel to use updated packages.


In [4]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, ysfitsutilpy, ysphotutilpy, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module ysfitsutilpy is installed
**** module ysphotutilpy is not installed


ERROR: Could not find a version that satisfies the requirement astroscrappy>=1.1.1 (from ysphotutilpy) (from versions: 1.0, 1.0.1, 1.0.2, 1.0.3, 1.0.5, 1.0.6, 1.0.7, 1.0.8, 1.1.0)
ERROR: No matching distribution found for astroscrappy>=1.1.1


CalledProcessError: Command '['/home/guitar79/anaconda3/envs/astro_Python_ubuntu_env/bin/python', '-m', 'pip', 'install', 'ysphotutilpy', '-q']' returned non-zero exit status 1.

### import modules

In [3]:
import os, shutil
from glob import glob
from pathlib import Path
import numpy as np

from astropy.io import fits

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import astro_utilities
import Python_utilities


In [4]:
BASEDIR = Path("/mnt/OBS_data") 
BASEDIR = Path("/mnt/Rdata/CCD_obs") 
#BASEDIR = Path("/mnt/OBS_data") 
DOINGDIR = Path( BASEDIR/ astro_utilities.CCD_obs_raw_dir)
#DOINGDIR = Path( BASEDIR/ astro_utilities.CCD_NEW_dir )
DOINGDIRs = sorted(Python_utilities.getFullnameListOfallsubDirs(DOINGDIR))
#print ("DOINGDIRs: ", format(DOINGDIRs))
print ("len(DOINGDIRs): ", format(len(DOINGDIRs)))

len(DOINGDIRs):  166


In [ ]:
for DOINGDIR in DOINGDIRs :
    DOINGDIR = Path(DOINGDIR)
    print("DOINGDIR", DOINGDIR)
    fits_in_dir = sorted(list(DOINGDIR.glob('*.fit*')))
    #print("fits_in_dir", fits_in_dir)
    print("len(fits_in_dir)", len(fits_in_dir))

In [6]:

if len(fits_in_dir) == 0 :
    print(f"There is no fits fils in {DOINGDIR}")
    pass
else : 
    #try:
    print(f"Starting: {str(DOINGDIR.parts[-1])}")
    summary = None 
    summary = yfu.make_summary(DOINGDIR/"*.fit*",
                #output = save_fpath,
                verbose = True
                )
    print("summary: ", summary)
    print("type(summary): ", type(summary))

    for _, row in summary.iterrows():
        # 파일명 출력
        print (row["file"])
        fpath = Path(row["file"])
        new_fpath = Path(f"{fpath.parents[0]}/{fpath.stem}_clean.fit")
        # fits hedaer 에 있는 wcs 정보를 지운다
        yfu.wcsremove(fpath, 
                    additional_keys=["COMMENT"],
                    verbose=True,
                    output=new_fpath,
                    ccddata=False,
                    overwrite=True)
        if new_fpath.exists() \
            and fpath.exists():
            print("rename", f"{str(new_fpath)}", f"{str(fpath)}")
            #os.rename(f"{str(new_fpath)}", f"{str(fpath)}")
            shutil.move(f"{str(new_fpath)}", f"{str(fpath)}")
                
# except Exception as err :
#     print("X"*60)
#     print('{0}'.format(err))

Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/
No FITS file found.
There is no fits fils in /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin
Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73//
No FITS file found.
There is no fits fils in /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73
Starting...
/mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73/21P_Light_-_2018-09-10_-_FSQ106ED-x73_QSI683ws_-_1bin///
All 68 keywords (guessed from /mnt/Rdata/CCD_obs/CCD_new_files/QSI683ws_1bin/Light_FSQ106ED-x73/21P_Light_-_2018-09-10_-_FSQ106ED-x73_QSI683ws_-_1bin/21P_Light_B_2018-09-10-16-41-60_100sec_FSQ106ED-x73_QSI683ws_-20C_1bin.fit) will be loaded.


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().